# Bloom — Jaren's instruction dataset
Continue the latest published Bloom base with response-only supervised fine-tuning. Upload bloom_instruction_expanded.json from your phone or place it in MyDrive/centauri-bloom-data/. The dataset stays in your runtime/Drive; it is not uploaded to GitHub or HF. Only run with permission to train and publish resulting weights. Dataset provenance and factual accuracy require further review.
Choose a GPU. Run setup/configuration, authorize with the password widget, then load data and train. The final cell submits a Hugging Face PR automatically, never commits directly to main. If upload fails, rerun only the last cell; the weights stay in Drive. Use User: ... newline Assistant: at inference. This is an experimental candidate, not a verified general assistant.

In [ ]:
# 1. Clone Centauri Bloom (main) and install dependencies
import os, sys, subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = Path('/kaggle/working').is_dir()
WORKSPACE = Path('/content') if IN_COLAB else (Path('/kaggle/working') if IN_KAGGLE else Path.cwd())
if Path('pyproject.toml').is_file() and Path('bloom/model.py').is_file():
    REPO = Path.cwd()
else:
    REPO = WORKSPACE / 'centauri-bloom'
    if not (REPO / 'pyproject.toml').is_file():
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/InterstellarNerd8760/centauri-bloom.git', str(REPO)], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tokenizers', 'datasets'], check=True)
subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True)

In [ ]:
# 2. Configure run & choose persistent output
import json, torch
from pathlib import Path

from datetime import datetime, timezone
RUN_NAME = 'bloom-jaren-instruct-' + datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
TINY_SMOKE_TEST = False        # Set True to test locally on CPU in a few seconds
HF_REPO_ID = 'Centauri-Bloom/Bloom-Text-0.1'

STEPS = 1500                  # Additional optimizer steps; adjust after evaluation
BATCH_SIZE = 16
GRAD_ACCUM = 2
LEARNING_RATE = 5e-5
WARMUP_STEPS = 50
SAVE_EVERY = 250
EVAL_BATCHES = 32
SEED = 47

if TINY_SMOKE_TEST:
    raise ValueError('Use the original notebook for a fresh CPU smoke test.')

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = Path('/kaggle/working').is_dir()
SAVE_TO_DRIVE = IN_COLAB
if SAVE_TO_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = Path('/content/drive/MyDrive/centauri-bloom-runs') / RUN_NAME
elif IN_KAGGLE:
    OUT_DIR = Path('/kaggle/working/bloom-runs') / RUN_NAME
else:
    OUT_DIR = Path(os.environ.get('BLOOM_OUTPUT_DIR', str(REPO / 'runs'))) / RUN_NAME
OUT_DIR.mkdir(parents=True, exist_ok=False)

DEVICE = 'cuda' if torch.cuda.is_available() and not TINY_SMOKE_TEST else 'cpu'
if DEVICE != 'cuda':
    raise RuntimeError('Select an authorized GPU runtime before training.')
print('Device:', DEVICE, '| Output dir:', OUT_DIR)
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 3. Authenticate once using a tap-friendly password field
from huggingface_hub import HfApi
from IPython.display import display
import ipywidgets as widgets
from google.colab import output as colab_output
colab_output.enable_custom_widget_manager()
AUTH_API = None
token_field = widgets.Password(description='HF token:', placeholder='Paste write token')
authorize = widgets.Button(description='Authorize upload', button_style='success')
auth_out = widgets.Output()
def authenticate(_):
    global AUTH_API
    with auth_out:
        auth_out.clear_output()
        token = token_field.value.strip()
        if not token:
            print('Paste your Hugging Face write token first.')
            return
        try:
            client = HfApi(token=token)
            client.whoami()
            AUTH_API = client
            token_field.value = ''
            authorize.disabled = True
            print('Authenticated. Run all remaining cells in order.')
            print('Publication mode: automatic candidate PR')
        except Exception as e:
            print('Authentication failed:', type(e).__name__)
authorize.on_click(authenticate)
display(token_field, authorize, auth_out)


In [ ]:
# 4. Restore parent and build a small original instruction corpus
import hashlib, subprocess, random
from tokenizers import Tokenizer
from huggingface_hub import HfApi, snapshot_download
import shutil

if AUTH_API is None:
    raise RuntimeError('Authorize upload in the previous cell first.')
PARENT_REVISION = AUTH_API.model_info(HF_REPO_ID).sha
if not PARENT_REVISION:
    raise RuntimeError('Could not resolve parent revision')
PARENT_DIR = Path(snapshot_download(repo_id=HF_REPO_ID, revision=PARENT_REVISION,
    allow_patterns=['config.json','tokenizer.json','model.safetensors','metrics.json']))
parent_metrics = json.loads((PARENT_DIR / 'metrics.json').read_text())
cfg_raw = json.loads((PARENT_DIR / 'config.json').read_text())
VOCAB_SIZE = cfg_raw['vocab_size']
CONTEXT = cfg_raw['context']
tok_path = OUT_DIR / 'tokenizer.json'
shutil.copyfile(PARENT_DIR / 'tokenizer.json', tok_path)
tokenizer = Tokenizer.from_file(str(tok_path))
if tokenizer.get_vocab_size() != VOCAB_SIZE:
    raise ValueError('Parent tokenizer/config mismatch')
eot_id = tokenizer.token_to_id('<|endoftext|>')
if eot_id is None:
    raise ValueError('Missing original end-of-text token')
print('Parent revision:', PARENT_REVISION, '| original vocab:', VOCAB_SIZE)
print('Reported parent training step:', parent_metrics.get('step'))

# Pin the exact contributor file supplied by Weston.
import re, unicodedata
from google.colab import files
EXPECTED_SHA = '2b9f251e0c0ae172e867845e9accaed7a2d59af16b047eb24e0c4176f62b60da'
data_dir=Path('/content/drive/MyDrive/centauri-bloom-data')
data_dir.mkdir(parents=True,exist_ok=True)
data_path=data_dir/'bloom_instruction_expanded.json'
if not data_path.exists():
    print('Choose bloom_instruction_expanded.json from Files.')
    uploaded=files.upload()
    if 'bloom_instruction_expanded.json' not in uploaded:
        raise ValueError('Choose the original JSON file.')
    raw=uploaded['bloom_instruction_expanded.json']
    if hashlib.sha256(raw).hexdigest()!=EXPECTED_SHA:
        raise ValueError('Dataset SHA-256 differs from the reviewed input.')
    data_path.write_bytes(raw)
raw=data_path.read_bytes()
if hashlib.sha256(raw).hexdigest()!=EXPECTED_SHA:
    raise ValueError('Dataset SHA-256 differs from the reviewed input.')
rows=json.loads(raw)
if not isinstance(rows,list):
    raise ValueError('Expected JSON array')
def normalize(q):
    return ' '.join(unicodedata.normalize('NFKC',q).casefold().split()).rstrip(' .!?')
pairs=[]
seen=set()
for row in rows:
    q,a=row.get('prompt'),row.get('response')
    if not isinstance(q,str) or not isinstance(a,str) or not q.strip() or not a.strip():
        continue
    q,a=q.strip(),a.strip()
    key=(normalize(q),a)
    if key not in seen:
        pairs.append((q,a))
        seen.add(key)
train_pairs,val_pairs=[],[]
for q,a in pairs:
    digest=hashlib.sha256(normalize(q).encode()).hexdigest()
    (val_pairs if int(digest[:8],16)%10==0 else train_pairs).append((q,a))
random.Random(SEED).shuffle(train_pairs)
random.Random(SEED).shuffle(val_pairs)
print('Source rows:',len(rows),'Deduplicated:',len(pairs),'Train:',len(train_pairs),'Validation:',len(val_pairs))
(OUT_DIR/'data-provenance.json').write_text(json.dumps({'recipe':'contributor prompt-response SFT','dataset':'Jaren contribution','source_filename':data_path.name,'source_sha256':EXPECTED_SHA,'source_count':len(rows),'deduplicated_count':len(pairs),'train_count':len(train_pairs),'validation_count':len(val_pairs),'split':'normalized prompt hash, 10% held out','note':'User supplied for training; redistribution license and full factual accuracy not independently verified. No raw dataset uploaded.'},indent=2))


In [ ]:
# 5. Pack windows and load validated parent weights
import numpy as np
from torch.utils.data import TensorDataset, DataLoader
from bloom.model import Bloom, Config
from bloom.checkpoint import config_hash, save_weights, load_weights

def encode_pairs(pairs):
    xs, ys = [], []
    for question, answer in pairs:
        prefix = tokenizer.encode('User: '+question+'\nAssistant:').ids
        response = tokenizer.encode(' '+answer).ids+[eot_id]
        if len(prefix)+len(response)>CONTEXT+1 or not prefix:
            continue
        ids=prefix+response
        x=ids[:-1]
        y=[-100]*(len(prefix)-1)+response
        pad=CONTEXT-len(x)
        xs.append(x+[eot_id]*pad)
        ys.append(y+[-100]*pad)
    if not xs:
        raise RuntimeError('No examples fit context')
    return TensorDataset(torch.tensor(xs,dtype=torch.long),torch.tensor(ys,dtype=torch.long))
train_ds=encode_pairs(train_pairs)
val_ds=encode_pairs(val_pairs)
def collate(batch):
    x=torch.stack([b[0] for b in batch])
    y=torch.stack([b[1] for b in batch])
    # Trim trailing padding for GPU efficiency; each row ends in a supervised EOT.
    length=int(torch.where(y!=-100)[1].max().item())+1
    return x[:,:length].contiguous(), y[:,:length].contiguous()
train_loader=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,drop_last=True,collate_fn=collate)
val_loader=DataLoader(val_ds,batch_size=BATCH_SIZE,shuffle=False,collate_fn=collate)
print('Encoded train/validation:',len(train_ds),len(val_ds))
if not len(train_loader):
    raise RuntimeError('Too few training examples for batch size')
config = Config(
    vocab_size=tokenizer.get_vocab_size(),
    context=CONTEXT,
    width=cfg_raw['width'],
    heads=cfg_raw['heads'],
    layers=cfg_raw['layers'],
)
torch.manual_seed(SEED)
model = Bloom(config)
if parent_metrics.get('config_sha256') != config_hash(config):
    raise ValueError('Parent configuration hash mismatch')
parent_sha = parent_metrics['weights_sha256']
model.load_state_dict(load_weights(PARENT_DIR / 'model.safetensors', parent_sha, model.state_dict()), strict=True)
model.to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f'Restored Bloom: {n_params:,} parameters | config_sha256: {config_hash(config)}')

In [ ]:
# 6. Baseline generation
def generate(prompt, max_new_tokens=128, temperature=0.8, top_k=40):
    model.eval()
    ids = tokenizer.encode(prompt).ids or [eot_id]
    idx = torch.tensor([ids], dtype=torch.long, device=DEVICE)
    with torch.no_grad():
        for _ in range(max_new_tokens):
            ctx = idx[:, -config.context:]
            logits = model(ctx)[:, -1, :] / max(temperature, 1e-5)
            if top_k:
                vals, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < vals[:, [-1]]] = float('-inf')
            probs = torch.softmax(logits, dim=-1)
            nxt = torch.multinomial(probs, num_samples=1)
            if nxt.item() == eot_id:
                break
            idx = torch.cat([idx, nxt], dim=1)
    return tokenizer.decode(idx[0].tolist())

PROMPTS=['Hello, how are you?', 'What is Earth?', 'Explain why the sky is blue.', 'What is 12 times 5?', 'Write a Python function that adds two numbers.', 'Write a short story about a robot.']
def samples():
    out={}
    for i,p in enumerate(PROMPTS):
        torch.manual_seed(SEED+i)
        out[p]=generate('User: '+p+'\nAssistant:',max_new_tokens=64,temperature=0.6,top_k=20)
    return out
parent_samples=samples()
print(json.dumps(parent_samples,indent=2))


In [ ]:
# 7. Train with periodic SafeTensors checkpoints saved to Google Drive
import math, time
from itertools import islice
from torch.nn import functional as F

def evaluate(m, loader, device, max_batches):
    m.eval()
    total, count = 0.0, 0
    with torch.no_grad():
        for x, y in islice(loader, max_batches):
            x, y = x.to(device), y.to(device)
            with torch.amp.autocast('cuda', dtype=torch.float16, enabled=(device == 'cuda')):
                logits = m(x)
                loss = F.cross_entropy(logits.flatten(0, 1), y.flatten(), reduction='sum')
            total += loss.item()
            count += (y != -100).sum().item()
    if not count or not math.isfinite(total):
        raise RuntimeError('Nonfinite or empty validation')
    return total / count, count

def save_ckpt(step, val_loss, train_loss, elapsed, tokens_seen):
    cpu_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
    ckpt_file = OUT_DIR / f'step-{step:06d}.safetensors'
    sha = save_weights(ckpt_file, cpu_state, cpu_state)
    latest_file = OUT_DIR / 'latest.safetensors'
    save_weights(latest_file, cpu_state, cpu_state)
    (OUT_DIR / 'config.json').write_text(json.dumps(config.__dict__, indent=2, sort_keys=True) + '\n')
    meta = {
        'step': int(parent_metrics.get('step', 0)) + step,
        'step_scope': 'attempted optimizer steps; see optimizer_updates',
        'additional_steps': step,
        'parent_revision': PARENT_REVISION,
        'parent_weights_sha256': parent_sha,
        'parent_reported_train_tokens': parent_metrics.get('train_tokens'),
        'parent_val_loss_same_windows': round(val_before, 4),
        'optimizer_state_restored': False,
        'parameters': n_params,
        'config_sha256': config_hash(config),
        'weights_sha256': sha,
        'train_tokens': input_tokens_seen,
        'train_tokens_scope': 'this continuation only; non-padding input tokens',
        'supervised_target_tokens': tokens_seen,
        'optimizer_updates': optimizer_updates,
        'cumulative_train_tokens': (parent_total + input_tokens_seen) if parent_total is not None else None,
        'cumulative_train_tokens_scope': 'reported prior pretraining target tokens plus non-padding instruction input tokens; repeated exposures included',
        'train_examples': len(train_ds),
        'validation_examples': len(val_ds),
        'last_train_loss': round(train_loss, 4),
        'val_loss': round(val_loss, 4),
        'elapsed_seconds': round(elapsed, 2),
        'use_tinystories': False,
        'device': DEVICE,
        'run_name': RUN_NAME,
        'dataset': 'Jaren instruction expanded',
        'training_track': 'instruction-tuning',
        'data_provenance_sha256': hashlib.sha256((OUT_DIR/'data-provenance.json').read_bytes()).hexdigest(),
        'prompt_format': 'User: {prompt}\\nAssistant:',
        'seed': SEED,
        'learning_rate': LEARNING_RATE,
        'batch_size': BATCH_SIZE,
        'grad_accum': GRAD_ACCUM,
        'code_revision': subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip(),
    }
    (OUT_DIR / 'metrics.json').write_text(json.dumps(meta, indent=2, sort_keys=True) + '\n')
    # Private run state; never load untrusted optimizer pickle files.
    torch.save({'optimizer': optimizer.state_dict(), 'scaler': scaler.state_dict(),
                'step': step, 'tokens_seen': tokens_seen, 'parent_revision': PARENT_REVISION},
               OUT_DIR / 'local-optimizer-state.pt')
    return sha

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.1, betas=(0.9, 0.95))
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE == 'cuda'))

def lr_for_step(step):
    if step < WARMUP_STEPS:
        return LEARNING_RATE * (step + 1) / max(1, WARMUP_STEPS)
    progress = (step - WARMUP_STEPS) / max(1, STEPS - WARMUP_STEPS)
    return LEARNING_RATE * (0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * progress)))

val_before, val_tokens = evaluate(model, val_loader, DEVICE, EVAL_BATCHES)
print(f'Initial val loss: {val_before:.4f} ({val_tokens} tokens)')

it = iter(train_loader)
tokens_seen = 0
input_tokens_seen = 0
optimizer_updates = 0
parent_total=parent_metrics.get('cumulative_train_tokens')
if parent_total is None and parent_metrics.get('train_tokens_scope') != 'this continuation only' and not str(parent_metrics.get('train_tokens_scope','')).startswith('this continuation'):
    parent_total=parent_metrics.get('train_tokens')
if parent_total is None and parent_metrics.get('step')==4500 and isinstance(parent_metrics.get('parent_reported_train_tokens'),int) and isinstance(parent_metrics.get('train_tokens'),int):
    parent_total=parent_metrics['parent_reported_train_tokens']+parent_metrics['train_tokens']
if not isinstance(parent_total,int) or parent_total<0:
    parent_total=None
print('Parent reported lifetime token exposures:',parent_total)
t0 = time.perf_counter()
last_loss = val_before

for step in range(1, STEPS + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    for pg in optimizer.param_groups:
        pg['lr'] = lr_for_step(step - 1)
    step_loss = 0.0
    for _ in range(GRAD_ACCUM):
        batch = next(it, None)
        if batch is None:
            it = iter(train_loader)
            batch = next(it)
        x, y = batch[0].to(DEVICE), batch[1].to(DEVICE)
        with torch.amp.autocast('cuda', dtype=torch.float16, enabled=(DEVICE == 'cuda')):
            loss = F.cross_entropy(model(x).flatten(0, 1), y.flatten()) / GRAD_ACCUM
        scaler.scale(loss).backward()
        step_loss += loss.item()
        tokens_seen += (y != -100).sum().item()
        positions=torch.arange(y.shape[1],device=y.device).expand_as(y)
        input_tokens_seen += ((positions.masked_fill(y==-100,-1).max(dim=1).values)+1).sum().item()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    old_scale=scaler.get_scale()
    scaler.step(optimizer)
    scaler.update()
    optimizer_updates += int(scaler.get_scale()>=old_scale)
    last_loss = step_loss

    if step == 1 or step % 100 == 0 or step == STEPS:
        dt = time.perf_counter() - t0
        print(f'step {step:4d}/{STEPS} | loss {last_loss:.4f} | {tokens_seen/max(dt,1e-6):,.0f} tok/s | {dt:.1f}s')
    if step % SAVE_EVERY == 0 or step == STEPS:
        vloss, _ = evaluate(model, val_loader, DEVICE, EVAL_BATCHES)
        sha = save_ckpt(step, vloss, last_loss, time.perf_counter() - t0, tokens_seen)
        print(f'  -> Saved checkpoint step {step} | val_loss {vloss:.4f} | sha256 {sha[:12]}...')

val_after, _ = evaluate(model, val_loader, DEVICE, EVAL_BATCHES)
print(f'Done in {time.perf_counter() - t0:.1f}s | Val loss: {val_before:.4f} -> {val_after:.4f}')
candidate_samples=samples()
report={'parent_revision':PARENT_REVISION,'parent_instruction_loss':val_before,'candidate_instruction_loss':val_after,'parent_samples':parent_samples,'candidate_samples':candidate_samples,'note':'Fixed qualitative prompts, not scored benchmarks; possible prior training overlap. Instruction loss not comparable to pretraining loss.'}
(OUT_DIR/'evaluation.json').write_text(json.dumps(report,indent=2))
metrics=json.loads((OUT_DIR/'metrics.json').read_text())
metrics['evaluation_sha256']=hashlib.sha256((OUT_DIR/'evaluation.json').read_bytes()).hexdigest()
metrics['validation_loss_scope']='response targets only, held-out contributor prompts'
(OUT_DIR/'metrics.json').write_text(json.dumps(metrics,indent=2,sort_keys=True)+'\n')
print(json.dumps(report,indent=2))
print(json.dumps(metrics,indent=2))


In [ ]:
# 8. Automatic PR submission, same successful method as recovery
import os
from huggingface_hub import CommitOperationAdd, constants
os.environ['HF_HUB_DISABLE_XET']='1'
constants.HF_HUB_DISABLE_XET=True
metrics=json.loads((OUT_DIR/'metrics.json').read_text())
if metrics['additional_steps']!=STEPS:
    raise RuntimeError('Training incomplete')
weights=OUT_DIR/f'step-{STEPS:06d}.safetensors'
for path,key in [(weights,'weights_sha256'),(OUT_DIR/'data-provenance.json','data_provenance_sha256'),(OUT_DIR/'evaluation.json','evaluation_sha256')]:
    if hashlib.sha256(path.read_bytes()).hexdigest()!=metrics[key]:
        raise RuntimeError('Hash mismatch: '+str(path))
receipt=OUT_DIR/'upload-result.json'
if receipt.exists():
    print('Already submitted:',receipt.read_text())
else:
    if AUTH_API.model_info(HF_REPO_ID).sha!=PARENT_REVISION:
        raise RuntimeError('Published parent changed; preserve candidate and review before retrying.')
    paths={'model.safetensors':weights,**{name:OUT_DIR/name for name in ['config.json','tokenizer.json','metrics.json','data-provenance.json','evaluation.json']}}
    operations=[CommitOperationAdd(path_in_repo=name,path_or_fileobj=str(path)) for name,path in paths.items()]
    print('Submitting candidate PR...',flush=True)
    result=AUTH_API.create_commit(repo_id=HF_REPO_ID,operations=operations,revision='main',parent_commit=PARENT_REVISION,create_pr=True,commit_message=f'Bloom Jaren instruction tuning {RUN_NAME}')
    receipt.write_text(json.dumps({'commit_url':str(result.commit_url),'pr_url':str(result.pr_url)},indent=2))
    print('UPLOAD COMPLETE',receipt.read_text())
print('Candidate saved:',OUT_DIR)
